# Representation Analysis: Mechanistic Evidence for Distillation Hypothesis

This notebook provides deep, representation-level analyses to investigate the "distillation hypothesis" - that negation understanding is compressed from BERT's layers 7/9 into DistilBERT's Layer 3.

## Core Analyses

1. **Representational Similarity Matrices (RSMs)** - Visualize how negation changes representations at each layer
2. **Probing the Probe** - Ablation studies to understand which neurons encode negation
3. **Cross-Model Consistency** - Canonical Correlation Analysis (CCA) between BERT and DistilBERT layers

## Prerequisites
Run this notebook **after** completing:
- `07_sweep_*.ipynb` (trained probes)
- `09_transfer_evaluation.ipynb` (transfer metrics)

## Key Outputs
- RSM heatmaps showing negation-induced representation changes
- Neuron ablation results identifying critical negation-encoding neurons
- CCA correlation matrices showing BERT→DistilBERT layer alignment
- Publication-ready figures for the distillation pathway


## 1. Setup & Configuration


In [ ]:
#@title **Configure Environment** { display-mode: "form" }
import os
import psutil

# Check resources
print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)

# RAM check
ram_gb = psutil.virtual_memory().total / 1e9
print(f"Available RAM: {ram_gb:.1f} GB")

if ram_gb > 25:
    print("  High-RAM mode detected")
    BATCH_SIZE = 64
else:
    print("  Standard RAM mode")
    BATCH_SIZE = 32

# GPU check
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    DEVICE = "cuda"
else:
    print("No GPU detected, using CPU")
    DEVICE = "cpu"
    BATCH_SIZE = 16

print(f"\nConfiguration:")
print(f"  Device: {DEVICE}")
print(f"  Batch size: {BATCH_SIZE}")
print("=" * 60)


In [ ]:
# Mount Google Drive (for Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Running locally")


In [ ]:
# Clone or pull the repository (Colab only)
if IN_COLAB:
    !git clone https://github.com/TheMattWang/Negation-Origin-Tracing.git 2>/dev/null || (cd Negation-Origin-Tracing && git pull)
    %cd Negation-Origin-Tracing


In [ ]:
# Install dependencies
%pip install -q torch lightning transformers datasets pandas pyarrow matplotlib seaborn scikit-learn tqdm scipy

print("Dependencies installed. If this is the first run, restart the runtime now.")


In [ ]:
# Set paths
import os
from datetime import datetime

if IN_COLAB:
    DRIVE_PATH = '/content/drive/MyDrive/NOT_results'
else:
    DRIVE_PATH = '../experiments'

OUTPUT_DIR = os.path.join(DRIVE_PATH, 'representation_analysis')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Results from sweeps: {DRIVE_PATH}")
print(f"Output directory: {OUTPUT_DIR}")


In [ ]:
# Core imports
import json
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.cross_decomposition import CCA

# Publication-quality settings
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'font.family': 'sans-serif',
    'axes.grid': True,
    'grid.alpha': 0.3,
})
sns.set_style("whitegrid")

# Color scheme
COLORS = {
    'anchor': '#3498db',      # Blue
    'negated': '#e74c3c',     # Red
    'layer3': '#f39c12',      # Gold (distillation target)
    'bert': '#9b59b6',        # Purple
    'distilbert': '#2ecc71',  # Green
    'highlight': '#f1c40f',   # Bright yellow
}

print("Imports and visualization settings configured.")


## 2. Load Models and Data


In [ ]:
# Load DistilBERT and BERT models
print("Loading models...")

# DistilBERT (6 layers)
DISTILBERT_NAME = "distilbert-base-uncased"
distilbert_tokenizer = AutoTokenizer.from_pretrained(DISTILBERT_NAME)
distilbert_model = AutoModel.from_pretrained(DISTILBERT_NAME)
distilbert_model = distilbert_model.to(DEVICE)
distilbert_model.eval()

print(f"DistilBERT loaded: {distilbert_model.config.num_hidden_layers} layers, {distilbert_model.config.hidden_size} hidden size")

# BERT-base (12 layers)
BERT_NAME = "bert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_NAME)
bert_model = AutoModel.from_pretrained(BERT_NAME)
bert_model = bert_model.to(DEVICE)
bert_model.eval()

print(f"BERT loaded: {bert_model.config.num_hidden_layers} layers, {bert_model.config.hidden_size} hidden size")


In [ ]:
# Download negation dataset if needed
NEGATION_DATA_DIR = 'data/raw/negation'

if not os.path.exists(os.path.join(NEGATION_DATA_DIR, 'test.parquet')):
    print("Downloading JinaAI Negation Dataset...")
    !python src/data/download_negation.py --output_dir {NEGATION_DATA_DIR}
else:
    print("Negation dataset already exists!")

# Show dataset info
for split in ['train', 'test']:
    path = os.path.join(NEGATION_DATA_DIR, f'{split}.parquet')
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f"  {split}: {len(df)} examples")


In [ ]:
# Load negation dataset
import sys
sys.path.insert(0, '.')

from src.datasets.dataset import NegationTripletDataset
from torch.utils.data import DataLoader

# Use test split for evaluation
test_path = os.path.join(NEGATION_DATA_DIR, 'test.parquet')
if not os.path.exists(test_path):
    test_path = os.path.join(NEGATION_DATA_DIR, 'train.parquet')
    print("Using train split (test not found)")

negation_dataset = NegationTripletDataset(
    data_path=test_path,
    tokenizer=distilbert_tokenizer,
    max_length=128,
)

negation_loader = DataLoader(
    negation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
)

print(f"\nLoaded {len(negation_dataset)} negation pairs")


## 3. Feature Extraction Utilities


In [ ]:
def extract_all_layer_features(model, tokenizer, texts, pooling='mean', device=DEVICE):
    """
    Extract features from all layers for a list of texts.
    
    Args:
        model: Transformer model (BERT or DistilBERT)
        tokenizer: Corresponding tokenizer
        texts: List of strings
        pooling: 'cls' or 'mean'
        device: Device to use
    
    Returns:
        dict: {layer_idx: tensor of shape (num_texts, hidden_size)}
    """
    model.eval()
    num_layers = model.config.num_hidden_layers
    
    # Tokenize all texts
    encodings = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )
    input_ids = encodings['input_ids'].to(device)
    attention_mask = encodings['attention_mask'].to(device)
    
    # Extract features
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
    
    hidden_states = outputs.hidden_states  # Tuple of (batch, seq_len, hidden)
    
    layer_features = {}
    for layer_idx in range(num_layers):
        # hidden_states[0] is embeddings, hidden_states[1] is layer 0, etc.
        layer_hidden = hidden_states[layer_idx + 1]
        
        if pooling == 'cls':
            pooled = layer_hidden[:, 0, :]
        elif pooling == 'mean':
            mask_expanded = attention_mask.unsqueeze(-1).expand(layer_hidden.size()).float()
            sum_hidden = torch.sum(layer_hidden * mask_expanded, dim=1)
            sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
            pooled = sum_hidden / sum_mask
        else:
            raise ValueError(f"Unknown pooling: {pooling}")
        
        layer_features[layer_idx] = pooled.cpu()
    
    return layer_features


def extract_paired_features(model, tokenizer, anchor_texts, negative_texts, pooling='mean', device=DEVICE):
    """
    Extract features for anchor-negative pairs from all layers.
    
    Returns:
        tuple: (anchor_features_dict, negative_features_dict)
    """
    anchor_features = extract_all_layer_features(model, tokenizer, anchor_texts, pooling, device)
    negative_features = extract_all_layer_features(model, tokenizer, negative_texts, pooling, device)
    return anchor_features, negative_features

print("Feature extraction utilities defined.")


In [ ]:
# Extract sample data for analysis
# Use a subset for visualization (full dataset for statistics)
N_SAMPLES = min(100, len(negation_dataset))  # Use 100 pairs for RSM visualization

print(f"Extracting features for {N_SAMPLES} pairs...")

# Get raw texts
anchor_texts = []
negative_texts = []

for i in range(N_SAMPLES):
    raw = negation_dataset.get_raw_texts(i)
    anchor_texts.append(raw['anchor'])
    negative_texts.append(raw['negative'])

print(f"Sample anchor: {anchor_texts[0][:60]}...")
print(f"Sample negative: {negative_texts[0][:60]}...")

# Extract features for DistilBERT
print("\nExtracting DistilBERT features...")
distilbert_anchor_features, distilbert_negative_features = extract_paired_features(
    distilbert_model, distilbert_tokenizer, anchor_texts, negative_texts, pooling='mean'
)

# Extract features for BERT
print("Extracting BERT features...")
bert_anchor_features, bert_negative_features = extract_paired_features(
    bert_model, bert_tokenizer, anchor_texts, negative_texts, pooling='mean'
)

print(f"\nDistilBERT: {len(distilbert_anchor_features)} layers")
print(f"BERT: {len(bert_anchor_features)} layers")


## 4. Analysis 1: Representational Similarity Matrices (RSMs)

For each layer, we compute the cosine similarity matrix between all sentence representations.

**Expected Pattern:**
- High similarity within anchor group (upper-left block)
- High similarity within negated group (lower-right block)
- **Lower similarity between anchor and negated blocks** (off-diagonal)
- The contrast should be **maximal at the layer where negation is most distinctly encoded** (Layer 3 for DistilBERT)


In [ ]:
def compute_rsm(features_a, features_b):
    """
    Compute Representational Similarity Matrix using cosine similarity.
    
    Args:
        features_a: Tensor of shape (N, hidden_size)
        features_b: Tensor of shape (M, hidden_size)
    
    Returns:
        Cosine similarity matrix of shape (N, M)
    """
    # Normalize
    a_norm = F.normalize(features_a, p=2, dim=1)
    b_norm = F.normalize(features_b, p=2, dim=1)
    
    # Compute cosine similarity
    sim_matrix = torch.mm(a_norm, b_norm.t())
    return sim_matrix.numpy()


def compute_block_rsm(anchor_features, negative_features):
    """
    Compute full RSM with block structure: [anchors, negatives].
    
    Returns:
        rsm: Full similarity matrix
        within_anchor: Mean similarity within anchor block
        within_negative: Mean similarity within negative block
        between_blocks: Mean similarity between anchor and negative blocks
    """
    # Concatenate features
    all_features = torch.cat([anchor_features, negative_features], dim=0)
    n_anchors = anchor_features.shape[0]
    
    # Compute full RSM
    rsm = compute_rsm(all_features, all_features)
    
    # Extract blocks
    anchor_block = rsm[:n_anchors, :n_anchors]
    negative_block = rsm[n_anchors:, n_anchors:]
    cross_block = rsm[:n_anchors, n_anchors:]
    
    # Compute mean similarities (excluding diagonal for within-block)
    within_anchor = np.mean(anchor_block[np.triu_indices(n_anchors, k=1)])
    within_negative = np.mean(negative_block[np.triu_indices(n_anchors, k=1)])
    between_blocks = np.mean(cross_block)
    
    return rsm, within_anchor, within_negative, between_blocks

print("RSM computation functions defined.")


In [ ]:
# Compute RSMs for all DistilBERT layers
print("Computing RSMs for DistilBERT layers...")

distilbert_rsm_stats = []

for layer_idx in range(6):
    rsm, within_a, within_n, between = compute_block_rsm(
        distilbert_anchor_features[layer_idx],
        distilbert_negative_features[layer_idx]
    )
    
    # Block contrast: how much lower is between-block similarity vs within-block
    avg_within = (within_a + within_n) / 2
    contrast = avg_within - between
    
    distilbert_rsm_stats.append({
        'layer': layer_idx,
        'within_anchor': within_a,
        'within_negative': within_n,
        'between_blocks': between,
        'contrast': contrast,
        'rsm': rsm
    })
    
    print(f"  Layer {layer_idx}: within={avg_within:.4f}, between={between:.4f}, contrast={contrast:.4f}")

# Find layer with maximum contrast
max_contrast_layer = max(distilbert_rsm_stats, key=lambda x: x['contrast'])['layer']
print(f"\n→ Maximum contrast at Layer {max_contrast_layer}")


In [ ]:
# Visualize RSMs for key layers
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for idx, layer_idx in enumerate(range(6)):
    ax = axes[idx // 3, idx % 3]
    rsm = distilbert_rsm_stats[layer_idx]['rsm']
    contrast = distilbert_rsm_stats[layer_idx]['contrast']
    
    # Plot heatmap
    im = ax.imshow(rsm, cmap='RdBu_r', vmin=0.5, vmax=1.0, aspect='auto')
    
    # Add block boundary
    n = N_SAMPLES
    ax.axhline(y=n-0.5, color='white', linewidth=2, linestyle='--')
    ax.axvline(x=n-0.5, color='white', linewidth=2, linestyle='--')
    
    # Highlight Layer 3
    title_color = COLORS['layer3'] if layer_idx == 3 else 'black'
    title_weight = 'bold' if layer_idx == 3 else 'normal'
    
    ax.set_title(f'Layer {layer_idx}\nContrast: {contrast:.4f}', 
                 fontsize=12, fontweight=title_weight, color=title_color)
    ax.set_xlabel('Sentence Index')
    ax.set_ylabel('Sentence Index')
    
    # Add block labels
    ax.text(n//2, -5, 'Anchors', ha='center', fontsize=9, color=COLORS['anchor'])
    ax.text(n + n//2, -5, 'Negated', ha='center', fontsize=9, color=COLORS['negated'])

# Add colorbar
fig.colorbar(im, ax=axes.ravel().tolist(), label='Cosine Similarity', shrink=0.6)

plt.suptitle('Representational Similarity Matrices: DistilBERT Layers\n(Lower between-block similarity = better negation encoding)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

# Save figure
save_path = os.path.join(OUTPUT_DIR, 'rsm_all_layers.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Saved: {save_path}")

plt.show()


In [ ]:
# Plot RSM contrast by layer
fig, ax = plt.subplots(figsize=(10, 6))

layers = [s['layer'] for s in distilbert_rsm_stats]
contrasts = [s['contrast'] for s in distilbert_rsm_stats]
within_sims = [(s['within_anchor'] + s['within_negative']) / 2 for s in distilbert_rsm_stats]
between_sims = [s['between_blocks'] for s in distilbert_rsm_stats]

# Plot bars
x = np.arange(len(layers))
width = 0.35

bars1 = ax.bar(x - width/2, within_sims, width, label='Within-Group Similarity', 
               color=COLORS['anchor'], alpha=0.8)
bars2 = ax.bar(x + width/2, between_sims, width, label='Between-Group Similarity', 
               color=COLORS['negated'], alpha=0.8)

# Highlight Layer 3
bars1[3].set_edgecolor(COLORS['layer3'])
bars1[3].set_linewidth(3)
bars2[3].set_edgecolor(COLORS['layer3'])
bars2[3].set_linewidth(3)

# Add contrast line
ax2 = ax.twinx()
ax2.plot(x, contrasts, 'ko-', linewidth=2, markersize=10, label='Contrast (Within - Between)')
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Mark maximum contrast
max_idx = np.argmax(contrasts)
ax2.annotate(f'Max Contrast\nLayer {max_idx}', 
             xy=(max_idx, contrasts[max_idx]),
             xytext=(max_idx + 0.5, contrasts[max_idx] + 0.01),
             arrowprops=dict(arrowstyle='->', color=COLORS['layer3'], lw=2),
             fontsize=10, fontweight='bold', color=COLORS['layer3'])

# Formatting
ax.set_xlabel('DistilBERT Layer', fontsize=12, fontweight='bold')
ax.set_ylabel('Cosine Similarity', fontsize=12, fontweight='bold')
ax2.set_ylabel('Contrast (Within - Between)', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(layers)
ax.set_ylim(0.7, 1.0)
ax.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.title('RSM Block Structure Analysis: Negation Encoding by Layer', fontsize=14, fontweight='bold')
plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, 'rsm_contrast_by_layer.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Saved: {save_path}")

plt.show()


## 5. Analysis 2: Probing the Probe - Neuron Ablation Study

This analysis investigates which neurons in Layer 3 are critical for negation encoding.

**Method:**
1. For sentences where the probe correctly flips prediction, zero-out specific neurons
2. Measure if flip accuracy drops dramatically
3. Compare with other layers as control

**Expected Result:**
- Layer 3 should be most sensitive to neuron ablation
- A small subset of neurons may be disproportionately important


In [ ]:
# Load trained probes for ablation study
from src.models import BaseModule

def load_probe_checkpoint(layer_idx, pooling='mean'):
    """Load trained probe from sweep results."""
    sweep_dir = os.path.join(DRIVE_PATH, f'sweep_{pooling}')
    
    # Try different checkpoint patterns
    patterns = [
        os.path.join(sweep_dir, 'checkpoints', f'layer{layer_idx}_{pooling}', 'best*.ckpt'),
        os.path.join(sweep_dir, f'layer_{layer_idx}_pooling_{pooling}', 'checkpoints', 'best*.ckpt'),
    ]
    
    for pattern in patterns:
        matches = glob.glob(pattern)
        if matches:
            checkpoint_path = matches[0]
            model = BaseModule.load_from_checkpoint(
                checkpoint_path,
                model_name=DISTILBERT_NAME,
                mode="probe",
                probe_layer=layer_idx,
                pooling_strategy=pooling,
            )
            model = model.to(DEVICE)
            model.eval()
            return model
    
    print(f"Warning: No checkpoint found for layer {layer_idx}, pooling {pooling}")
    return None

print("Probe loading function defined.")


In [ ]:
def ablation_experiment(model, layer_idx, anchor_features, negative_features, 
                        ablation_fraction=0.1, n_trials=10):
    """
    Perform neuron ablation experiment.
    
    Args:
        model: Trained probe model
        layer_idx: Layer being probed
        anchor_features: Anchor sentence features
        negative_features: Negative sentence features
        ablation_fraction: Fraction of neurons to ablate
        n_trials: Number of random ablation trials
    
    Returns:
        dict with baseline and ablated flip accuracies
    """
    probe = model.probes[f"layer_{layer_idx}"]
    hidden_size = anchor_features.shape[1]
    n_ablate = int(hidden_size * ablation_fraction)
    
    # Move features to device
    anchor_feat = anchor_features.to(DEVICE)
    negative_feat = negative_features.to(DEVICE)
    
    with torch.no_grad():
        # Baseline predictions
        anchor_logits = probe(anchor_feat)
        negative_logits = probe(negative_feat)
        
        anchor_preds = torch.argmax(anchor_logits, dim=-1)
        negative_preds = torch.argmax(negative_logits, dim=-1)
        
        baseline_flips = (anchor_preds != negative_preds).float()
        baseline_flip_acc = baseline_flips.mean().item()
        
        # Random ablation trials
        ablated_flip_accs = []
        
        for trial in range(n_trials):
            # Random neurons to ablate
            ablate_indices = np.random.choice(hidden_size, n_ablate, replace=False)
            
            # Create ablated features
            anchor_ablated = anchor_feat.clone()
            negative_ablated = negative_feat.clone()
            anchor_ablated[:, ablate_indices] = 0
            negative_ablated[:, ablate_indices] = 0
            
            # Get predictions
            anchor_logits_abl = probe(anchor_ablated)
            negative_logits_abl = probe(negative_ablated)
            
            anchor_preds_abl = torch.argmax(anchor_logits_abl, dim=-1)
            negative_preds_abl = torch.argmax(negative_logits_abl, dim=-1)
            
            ablated_flips = (anchor_preds_abl != negative_preds_abl).float()
            ablated_flip_accs.append(ablated_flips.mean().item())
    
    return {
        'layer': layer_idx,
        'baseline_flip_acc': baseline_flip_acc,
        'ablated_flip_acc_mean': np.mean(ablated_flip_accs),
        'ablated_flip_acc_std': np.std(ablated_flip_accs),
        'accuracy_drop': baseline_flip_acc - np.mean(ablated_flip_accs),
        'n_neurons_ablated': n_ablate,
    }

print("Ablation experiment function defined.")


In [ ]:
# Run ablation experiments for all layers
print("Running ablation experiments...")
print("(This may take a few minutes)\n")

ablation_results = []

for layer_idx in range(6):
    print(f"Layer {layer_idx}...", end=" ")
    
    # Load probe
    model = load_probe_checkpoint(layer_idx, pooling='mean')
    
    if model is None:
        print("SKIPPED (no checkpoint)")
        continue
    
    # Run ablation
    result = ablation_experiment(
        model, layer_idx,
        distilbert_anchor_features[layer_idx],
        distilbert_negative_features[layer_idx],
        ablation_fraction=0.1,
        n_trials=20
    )
    ablation_results.append(result)
    
    print(f"baseline={result['baseline_flip_acc']:.3f}, ablated={result['ablated_flip_acc_mean']:.3f} (±{result['ablated_flip_acc_std']:.3f})")
    
    # Clean up
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print("\nAblation experiments complete.")


In [ ]:
# Visualize ablation results
if ablation_results:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    layers = [r['layer'] for r in ablation_results]
    baseline_accs = [r['baseline_flip_acc'] for r in ablation_results]
    ablated_accs = [r['ablated_flip_acc_mean'] for r in ablation_results]
    ablated_stds = [r['ablated_flip_acc_std'] for r in ablation_results]
    drops = [r['accuracy_drop'] for r in ablation_results]
    
    x = np.arange(len(layers))
    width = 0.35
    
    # Plot bars
    bars1 = ax.bar(x - width/2, baseline_accs, width, label='Baseline Flip Accuracy',
                   color=COLORS['anchor'], alpha=0.8)
    bars2 = ax.bar(x + width/2, ablated_accs, width, label='After 10% Neuron Ablation',
                   color=COLORS['negated'], alpha=0.8, yerr=ablated_stds, capsize=5)
    
    # Highlight Layer 3
    if 3 in layers:
        idx_3 = layers.index(3)
        bars1[idx_3].set_edgecolor(COLORS['layer3'])
        bars1[idx_3].set_linewidth(3)
        bars2[idx_3].set_edgecolor(COLORS['layer3'])
        bars2[idx_3].set_linewidth(3)
    
    # Add accuracy drop annotations
    for i, (layer, drop) in enumerate(zip(layers, drops)):
        ax.annotate(f'Δ={drop:.3f}', xy=(i, max(baseline_accs[i], ablated_accs[i]) + 0.02),
                   ha='center', fontsize=9, fontweight='bold')
    
    ax.set_xlabel('DistilBERT Layer', fontsize=12, fontweight='bold')
    ax.set_ylabel('Flip Accuracy', fontsize=12, fontweight='bold')
    ax.set_title('Neuron Ablation Study: Impact on Negation Flip Detection', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(layers)
    ax.legend(loc='upper left')
    ax.set_ylim(0, 1.0)
    
    plt.tight_layout()
    
    save_path = os.path.join(OUTPUT_DIR, 'ablation_study.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Saved: {save_path}")
    
    plt.show()
else:
    print("No ablation results to visualize. Check that probe checkpoints exist.")


## 6. Analysis 3: LDA-based Negation Direction

Use Linear Discriminant Analysis to find the direction in activation space that best separates anchors from negatives.

**Method:**
1. Fit LDA on Layer 3 representations (anchors vs negatives)
2. Project all sentences onto the discriminant direction
3. Visualize separation quality

**Expected Result:**
- Clear separation between anchor and negative projections at Layer 3


In [ ]:
def compute_lda_separation(anchor_features, negative_features):
    """
    Compute LDA separation between anchors and negatives.
    
    Returns:
        dict with LDA model, projections, and separation metrics
    """
    # Prepare data
    X = torch.cat([anchor_features, negative_features], dim=0).numpy()
    y = np.array([0] * len(anchor_features) + [1] * len(negative_features))
    
    # Fit LDA
    lda = LinearDiscriminantAnalysis(n_components=1)
    X_lda = lda.fit_transform(X, y)
    
    # Compute separation metrics
    anchor_proj = X_lda[:len(anchor_features)].flatten()
    negative_proj = X_lda[len(anchor_features):].flatten()
    
    # Cohen's d effect size
    pooled_std = np.sqrt((np.var(anchor_proj) + np.var(negative_proj)) / 2)
    cohens_d = abs(np.mean(anchor_proj) - np.mean(negative_proj)) / pooled_std if pooled_std > 0 else 0
    
    # AUC for separation
    from sklearn.metrics import roc_auc_score
    auc = roc_auc_score(y, X_lda.flatten())
    if auc < 0.5:
        auc = 1 - auc  # Ensure AUC > 0.5
    
    return {
        'lda': lda,
        'anchor_projections': anchor_proj,
        'negative_projections': negative_proj,
        'cohens_d': cohens_d,
        'auc': auc,
        'explained_variance': lda.explained_variance_ratio_[0] if hasattr(lda, 'explained_variance_ratio_') else None,
    }

print("LDA separation function defined.")


In [ ]:
# Compute LDA separation for all layers
print("Computing LDA separation for all layers...")

lda_results = []

for layer_idx in range(6):
    result = compute_lda_separation(
        distilbert_anchor_features[layer_idx],
        distilbert_negative_features[layer_idx]
    )
    result['layer'] = layer_idx
    lda_results.append(result)
    
    print(f"  Layer {layer_idx}: Cohen's d = {result['cohens_d']:.3f}, AUC = {result['auc']:.3f}")

# Find best layer
best_lda_layer = max(lda_results, key=lambda x: x['cohens_d'])['layer']
print(f"\n→ Best LDA separation at Layer {best_lda_layer}")


In [ ]:
# Visualize LDA projections
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for idx, layer_idx in enumerate(range(6)):
    ax = axes[idx // 3, idx % 3]
    result = lda_results[layer_idx]
    
    # Plot histograms
    ax.hist(result['anchor_projections'], bins=30, alpha=0.6, 
            color=COLORS['anchor'], label='Anchors', density=True)
    ax.hist(result['negative_projections'], bins=30, alpha=0.6, 
            color=COLORS['negated'], label='Negated', density=True)
    
    # Highlight Layer 3
    title_color = COLORS['layer3'] if layer_idx == 3 else 'black'
    title_weight = 'bold' if layer_idx == 3 else 'normal'
    
    ax.set_title(f"Layer {layer_idx}\nCohen's d = {result['cohens_d']:.3f}", 
                 fontsize=12, fontweight=title_weight, color=title_color)
    ax.set_xlabel('LDA Projection')
    ax.set_ylabel('Density')
    ax.legend(loc='upper right')

plt.suptitle('LDA Projections: Anchor vs Negated Sentence Separation by Layer', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, 'lda_separation_all_layers.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Saved: {save_path}")

plt.show()


In [ ]:
# Summary plot: LDA metrics by layer
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

layers = [r['layer'] for r in lda_results]
cohens_ds = [r['cohens_d'] for r in lda_results]
aucs = [r['auc'] for r in lda_results]

# Cohen's d
bars1 = ax1.bar(layers, cohens_ds, color=COLORS['anchor'], alpha=0.8, edgecolor='black')
bars1[3].set_color(COLORS['layer3'])
bars1[3].set_edgecolor('black')
bars1[3].set_linewidth(2)

ax1.set_xlabel('DistilBERT Layer', fontsize=12, fontweight='bold')
ax1.set_ylabel("Cohen's d (Effect Size)", fontsize=12, fontweight='bold')
ax1.set_title("LDA Separation: Cohen's d by Layer", fontsize=14, fontweight='bold')
ax1.axhline(y=0.8, color='gray', linestyle='--', alpha=0.7, label='Large effect (0.8)')
ax1.legend()

# AUC
bars2 = ax2.bar(layers, aucs, color=COLORS['negated'], alpha=0.8, edgecolor='black')
bars2[3].set_color(COLORS['layer3'])
bars2[3].set_edgecolor('black')
bars2[3].set_linewidth(2)

ax2.set_xlabel('DistilBERT Layer', fontsize=12, fontweight='bold')
ax2.set_ylabel('AUC', fontsize=12, fontweight='bold')
ax2.set_title('LDA Separation: AUC by Layer', fontsize=14, fontweight='bold')
ax2.axhline(y=0.5, color='gray', linestyle='--', alpha=0.7, label='Random (0.5)')
ax2.set_ylim(0.4, 1.0)
ax2.legend()

plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, 'lda_metrics_by_layer.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Saved: {save_path}")

plt.show()


## 7. Analysis 4: Cross-Model Canonical Correlation Analysis (CCA)

This analysis tests the distillation hypothesis directly by measuring representational alignment between BERT and DistilBERT layers.

**Hypothesis:**
- DistilBERT Layer 3 should have highest correlation with BERT Layers 7/9
- This shows knowledge was transferred, not just that Layer 3 is independently good

**Method:**
- Compute CCA between each BERT layer and each DistilBERT layer
- Create correlation matrix showing alignment strength


In [ ]:
def compute_cca_correlation(features_a, features_b, n_components=10):
    """
    Compute CCA correlation between two sets of features.
    
    Args:
        features_a: Tensor of shape (N, hidden_size_a)
        features_b: Tensor of shape (N, hidden_size_b)
        n_components: Number of CCA components
    
    Returns:
        Mean canonical correlation
    """
    X = features_a.numpy()
    Y = features_b.numpy()
    
    # Limit components to valid range
    n_components = min(n_components, X.shape[0] - 1, X.shape[1], Y.shape[1])
    
    if n_components < 1:
        return 0.0
    
    # Fit CCA
    cca = CCA(n_components=n_components)
    try:
        X_c, Y_c = cca.fit_transform(X, Y)
        
        # Compute correlations for each component
        correlations = []
        for i in range(n_components):
            corr = np.corrcoef(X_c[:, i], Y_c[:, i])[0, 1]
            if not np.isnan(corr):
                correlations.append(abs(corr))
        
        return np.mean(correlations) if correlations else 0.0
    except Exception as e:
        print(f"CCA error: {e}")
        return 0.0

print("CCA correlation function defined.")


In [ ]:
# Compute CCA correlation matrix between BERT and DistilBERT layers
print("Computing CCA correlations between BERT and DistilBERT layers...")
print("(This may take a few minutes)\n")

# Combine anchor and negative features for more data
distilbert_all_features = {
    layer: torch.cat([distilbert_anchor_features[layer], distilbert_negative_features[layer]], dim=0)
    for layer in range(6)
}
bert_all_features = {
    layer: torch.cat([bert_anchor_features[layer], bert_negative_features[layer]], dim=0)
    for layer in range(12)
}

# Compute correlation matrix
cca_matrix = np.zeros((12, 6))  # BERT layers x DistilBERT layers

for bert_layer in tqdm(range(12), desc="BERT layers"):
    for distil_layer in range(6):
        corr = compute_cca_correlation(
            bert_all_features[bert_layer],
            distilbert_all_features[distil_layer],
            n_components=10
        )
        cca_matrix[bert_layer, distil_layer] = corr

print("\nCCA correlation matrix computed.")


In [ ]:
# Visualize CCA correlation matrix
fig, ax = plt.subplots(figsize=(10, 12))

# Create heatmap
im = ax.imshow(cca_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)

# Add colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.6)
cbar.set_label('Mean CCA Correlation', fontsize=12)

# Highlight expected distillation mappings
# DistilBERT 3 ← BERT 7/9
expected_mappings = [(7, 3), (9, 3)]  # (BERT layer, DistilBERT layer)

for bert_layer, distil_layer in expected_mappings:
    rect = plt.Rectangle((distil_layer - 0.5, bert_layer - 0.5), 1, 1, 
                          fill=False, edgecolor=COLORS['layer3'], linewidth=3)
    ax.add_patch(rect)

# Add text annotations for high correlations
for i in range(12):
    for j in range(6):
        text_color = 'white' if cca_matrix[i, j] > 0.5 else 'black'
        ax.text(j, i, f'{cca_matrix[i, j]:.2f}', ha='center', va='center', 
                fontsize=8, color=text_color)

# Formatting
ax.set_xlabel('DistilBERT Layer', fontsize=12, fontweight='bold')
ax.set_ylabel('BERT Layer', fontsize=12, fontweight='bold')
ax.set_xticks(range(6))
ax.set_yticks(range(12))
ax.set_title('Cross-Model CCA: BERT → DistilBERT Layer Alignment\n(Gold boxes: Expected distillation targets)', 
             fontsize=14, fontweight='bold')

plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, 'cca_correlation_matrix.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Saved: {save_path}")

plt.show()


In [ ]:
# "The Distillation Pathway" Figure
# Line plot showing CCA correlation for each DistilBERT layer across BERT layers

fig, ax = plt.subplots(figsize=(12, 6))

bert_layers = list(range(12))
colors = plt.cm.viridis(np.linspace(0, 1, 6))

for distil_layer in range(6):
    correlations = cca_matrix[:, distil_layer]
    
    # Highlight Layer 3 in gold
    if distil_layer == 3:
        ax.plot(bert_layers, correlations, 'o-', linewidth=3, markersize=10,
                color=COLORS['layer3'], label=f'DistilBERT Layer 3 (Target)', zorder=10)
    else:
        ax.plot(bert_layers, correlations, 'o-', linewidth=1.5, markersize=6,
                color=colors[distil_layer], alpha=0.6, label=f'DistilBERT Layer {distil_layer}')

# Highlight BERT negation layers (7, 9)
ax.axvspan(6.5, 9.5, alpha=0.15, color='red', label='BERT Negation Layers (7-9)')

# Mark peak for Layer 3
layer3_corrs = cca_matrix[:, 3]
peak_bert_layer = np.argmax(layer3_corrs)
ax.annotate(f'Peak: BERT {peak_bert_layer}\n→ DistilBERT 3',
            xy=(peak_bert_layer, layer3_corrs[peak_bert_layer]),
            xytext=(peak_bert_layer + 1.5, layer3_corrs[peak_bert_layer] + 0.05),
            arrowprops=dict(arrowstyle='->', color=COLORS['layer3'], lw=2),
            fontsize=11, fontweight='bold', color=COLORS['layer3'],
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor=COLORS['layer3']))

# Formatting
ax.set_xlabel('BERT Layer', fontsize=12, fontweight='bold')
ax.set_ylabel('CCA Correlation with DistilBERT', fontsize=12, fontweight='bold')
ax.set_title('The Distillation Pathway: BERT → DistilBERT Layer Mapping\n(Higher correlation = stronger representational alignment)', 
             fontsize=14, fontweight='bold')
ax.set_xticks(bert_layers)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.grid(True, alpha=0.3)

plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, 'distillation_pathway.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Saved: {save_path}")

plt.show()


## 8. Flip Consistency Analysis

Multi-panel figure for key layers showing:
1. Confidence Anchor vs Confidence Negative
2. Histogram of cosine similarity within negation pairs vs paraphrase control
3. RSM subset visualization


In [ ]:
def compute_pairwise_similarities(anchor_features, negative_features):
    """
    Compute cosine similarity between each anchor-negative pair.
    """
    # Normalize
    anchor_norm = F.normalize(anchor_features, p=2, dim=1)
    negative_norm = F.normalize(negative_features, p=2, dim=1)
    
    # Pairwise cosine similarity (diagonal of similarity matrix)
    similarities = (anchor_norm * negative_norm).sum(dim=1)
    return similarities.numpy()

# Compute pairwise similarities for all layers
pairwise_sims = {}
for layer_idx in range(6):
    sims = compute_pairwise_similarities(
        distilbert_anchor_features[layer_idx],
        distilbert_negative_features[layer_idx]
    )
    pairwise_sims[layer_idx] = sims
    print(f"Layer {layer_idx}: mean pairwise similarity = {np.mean(sims):.4f}")

print("\nPairwise similarities computed.")


In [ ]:
# Create "Flip Consistency Plot" - Multi-panel figure for Layer 3
key_layer = 3  # Distillation target

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Pairwise similarity histogram
ax1 = axes[0]
ax1.hist(pairwise_sims[key_layer], bins=30, color=COLORS['layer3'], alpha=0.8, edgecolor='black')
ax1.axvline(x=np.mean(pairwise_sims[key_layer]), color='red', linestyle='--', linewidth=2,
            label=f"Mean = {np.mean(pairwise_sims[key_layer]):.3f}")
ax1.set_xlabel('Cosine Similarity (Anchor-Negative)', fontsize=11, fontweight='bold')
ax1.set_ylabel('Count', fontsize=11, fontweight='bold')
ax1.set_title(f'Panel 1: Pairwise Similarity Distribution\n(Layer {key_layer})', fontsize=12, fontweight='bold')
ax1.legend()

# Panel 2: Similarity comparison across layers
ax2 = axes[1]
layer_means = [np.mean(pairwise_sims[i]) for i in range(6)]
layer_stds = [np.std(pairwise_sims[i]) for i in range(6)]

bars = ax2.bar(range(6), layer_means, yerr=layer_stds, capsize=5, 
               color=[COLORS['layer3'] if i == 3 else COLORS['anchor'] for i in range(6)],
               alpha=0.8, edgecolor='black')

ax2.set_xlabel('DistilBERT Layer', fontsize=11, fontweight='bold')
ax2.set_ylabel('Mean Pairwise Similarity', fontsize=11, fontweight='bold')
ax2.set_title('Panel 2: Negation Sensitivity by Layer\n(Lower = More Sensitive)', fontsize=12, fontweight='bold')

# Mark minimum
min_layer = np.argmin(layer_means)
ax2.annotate(f'Most Sensitive\nLayer {min_layer}',
             xy=(min_layer, layer_means[min_layer]),
             xytext=(min_layer + 0.5, layer_means[min_layer] - 0.02),
             arrowprops=dict(arrowstyle='->', color='red', lw=2),
             fontsize=10, fontweight='bold', color='red')

# Panel 3: RSM subset for key layer
ax3 = axes[2]
rsm_subset = distilbert_rsm_stats[key_layer]['rsm'][:30, :30]  # First 30 samples
im = ax3.imshow(rsm_subset, cmap='RdBu_r', vmin=0.7, vmax=1.0, aspect='auto')

# Add block boundary (assuming 15 anchors, 15 negatives in subset)
ax3.axhline(y=14.5, color='white', linewidth=2, linestyle='--')
ax3.axvline(x=14.5, color='white', linewidth=2, linestyle='--')

ax3.set_xlabel('Sentence Index', fontsize=11, fontweight='bold')
ax3.set_ylabel('Sentence Index', fontsize=11, fontweight='bold')
ax3.set_title(f'Panel 3: RSM Subset (Layer {key_layer})\n(Block structure visible)', fontsize=12, fontweight='bold')

# Add colorbar
plt.colorbar(im, ax=ax3, shrink=0.8, label='Cosine Similarity')

plt.suptitle('The Flip Consistency Plot: Layer 3 Negation Encoding', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

save_path = os.path.join(OUTPUT_DIR, 'flip_consistency_plot.png')
plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Saved: {save_path}")

plt.show()


## 9. Summary and Conclusions


In [ ]:

# Compile summary statistics
print("=" * 70)
print("REPRESENTATION ANALYSIS SUMMARY")
print("=" * 70)

print("\n1. RSM BLOCK STRUCTURE ANALYSIS")
print("-" * 40)
for stat in distilbert_rsm_stats:
    print(f"   Layer {stat['layer']}: contrast = {stat['contrast']:.4f}")
max_contrast_layer = max(distilbert_rsm_stats, key=lambda x: x['contrast'])['layer']
print(f"   → Maximum contrast at Layer {max_contrast_layer}")

print("\n2. LDA SEPARATION ANALYSIS")
print("-" * 40)
for result in lda_results:
    print(f"   Layer {result['layer']}: Cohen's d = {result['cohens_d']:.3f}, AUC = {result['auc']:.3f}")
best_lda = max(lda_results, key=lambda x: x['cohens_d'])
print(f"   → Best separation at Layer {best_lda['layer']} (d = {best_lda['cohens_d']:.3f})")

print("\n3. PAIRWISE SIMILARITY ANALYSIS")
print("-" * 40)
for layer_idx in range(6):
    print(f"   Layer {layer_idx}: mean = {np.mean(pairwise_sims[layer_idx]):.4f}")
min_sim_layer = np.argmin([np.mean(pairwise_sims[i]) for i in range(6)])
print(f"   → Most negation-sensitive at Layer {min_sim_layer}")

print("\n4. CCA CROSS-MODEL ALIGNMENT")
print("-" * 40)
# Find strongest BERT-DistilBERT alignments
for distil_layer in range(6):
    best_bert = np.argmax(cca_matrix[:, distil_layer])
    corr = cca_matrix[best_bert, distil_layer]
    print(f"   DistilBERT {distil_layer} ← BERT {best_bert} (corr = {corr:.3f})")

# Specific check for Layer 3
layer3_best_bert = np.argmax(cca_matrix[:, 3])
print(f"   → DistilBERT Layer 3 aligns best with BERT Layer {layer3_best_bert}")

print("\n" + "=" * 70)
print("DISTILLATION HYPOTHESIS EVIDENCE")
print("=" * 70)

# Evaluate evidence for distillation hypothesis
evidence_for = []
evidence_against = []

# Check if Layer 3 is consistently best
if max_contrast_layer == 3:
    evidence_for.append("RSM contrast peaks at Layer 3")
else:
    evidence_against.append(f"RSM contrast peaks at Layer {max_contrast_layer}, not 3")

if best_lda['layer'] == 3:
    evidence_for.append("LDA separation best at Layer 3")
else:
    evidence_against.append(f"LDA separation best at Layer {best_lda['layer']}, not 3")

if min_sim_layer == 3:
    evidence_for.append("Lowest pairwise similarity at Layer 3")
else:
    evidence_against.append(f"Lowest pairwise similarity at Layer {min_sim_layer}, not 3")

# Check BERT alignment
if layer3_best_bert in [7, 8, 9]:
    evidence_for.append(f"Layer 3 aligns with BERT Layer {layer3_best_bert} (expected: 7-9)")
else:
    evidence_against.append(f"Layer 3 aligns with BERT Layer {layer3_best_bert} (expected: 7-9)")

print("\nEvidence FOR distillation hypothesis:")
for e in evidence_for:
    print(f"  ✓ {e}")

print("\nEvidence AGAINST or NEUTRAL:")
for e in evidence_against:
    print(f"  ✗ {e}")

print("\n" + "=" * 70)

In [ ]:
# Save all results to JSON
results_summary = {
    'rsm_stats': [
        {
            'layer': s['layer'],
            'within_anchor': float(s['within_anchor']),
            'within_negative': float(s['within_negative']),
            'between_blocks': float(s['between_blocks']),
            'contrast': float(s['contrast']),
        }
        for s in distilbert_rsm_stats
    ],
    'lda_stats': [
        {
            'layer': r['layer'],
            'cohens_d': float(r['cohens_d']),
            'auc': float(r['auc']),
        }
        for r in lda_results
    ],
    'pairwise_similarity': {
        str(layer): {
            'mean': float(np.mean(sims)),
            'std': float(np.std(sims)),
        }
        for layer, sims in pairwise_sims.items()
    },
    'cca_matrix': cca_matrix.tolist(),
    'best_layers': {
        'rsm_contrast': int(max_contrast_layer),
        'lda_separation': int(best_lda['layer']),
        'pairwise_sensitivity': int(min_sim_layer),
        'bert_alignment_for_layer3': int(layer3_best_bert),
    },
    'n_samples': N_SAMPLES,
    'timestamp': datetime.now().isoformat(),
}

results_path = os.path.join(OUTPUT_DIR, 'representation_analysis_results.json')
with open(results_path, 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"Results saved to: {results_path}")
